In [26]:
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Tuple, Optional, Iterable
from datetime import datetime
from itertools import islice
import json
import csv

START_ELO = 500
WIN_POINTS = 5
LOSS_POINTS = -5

K_DEFAULT = 32.0        
ELO_SCALE = 400.0       

In [41]:
# ---------- Data model ----------

@dataclass
class RatingEvent:
    """
    Immutable snapshot of a single rating update for a player.

    Attributes:
        date: Match date; if unavailable, set to datetime.min.
        opponent: Opponent's display name.
        result: "W" for win, "L" for loss (from this player's perspective).
        delta: Rating change from this match (+5 for win, -5 for loss in this basic system).
        new_elo: Player's Elo immediately AFTER applying 'delta'.
        meta: Tuple of (tournament, surface, round) for light-weight metadata.
    """
    date: datetime
    opponent: str
    result: str        # "W" or "L"
    delta: int
    new_elo: int
    meta: Tuple[str, str, str] = field(default_factory=tuple)  # (tournament, surface, round)

@dataclass
class Player:
    """
    Player state tracked in-memory.

    Attributes:
        name: Display name as used in the CSV.
        elo: Current Elo (starts at START_ELO).
        wins: Number of matches won.
        losses: Number of matches lost.
        history: Chronological list of RatingEvent updates.
    """
    name: str
    elo: int = START_ELO
    wins: int = 0
    losses: int = 0
    history: List[RatingEvent] = field(default_factory=list)

Players = Dict[str, Player]
players_db: Players = {}



In [43]:
# ---------- Helpers ----------

def ensure_player(name: str) -> Player:
    """
    Ensure a Player entry exists for 'name' in the players_db.

    If the player does not exist, create a new Player initialized to START_ELO.

    Args:
        name: Player display name (used as the key).

    Returns:
        The Player object corresponding to 'name'.
    """
    name = name.strip()
    if not name:
        raise ValueError("Player name is empty after stripping.")
    if name not in players_db:
        players_db[name] = Player(name=name)
    return players_db[name]


def _first_n(iterable: Iterable, n: Optional[int]) -> Iterable:
    """
    Yield at most the first n items from an iterable; if n is None, yield all.

    Args:
        iterable: Any iterable (e.g., csv.DictReader).
        n: Optional limit; if None, do not limit.

    Returns:
        An iterator over up to n items.
    """
    return iterable if n is None else islice(iterable, n)


def parse_iso_date(s: str) -> Optional[datetime]:
    """
    Parse a date string from the CSV.
    Supports 'd/m/Y' (e.g. '5/01/2004') and ISO 'YYYY-MM-DD'.
    Returns None if parsing fails or string is empty.
    """
    s = (s or "").strip()
    if not s:
        return None
    # Try dd/mm/yyyy
    try:
        return datetime.strptime(s, "%d/%m/%Y")
    except ValueError:
        pass
    # Try ISO yyyy-mm-dd
    try:
        return datetime.strptime(s, "%Y-%m-%d")
    except ValueError:
        pass
    return None

def expected_score(r_a: float, r_b: float, scale: float = ELO_SCALE) -> float:
    """Probability Player A beats Player B, from Elo ratings r_a, r_b."""
    return 1.0 / (1.0 + 10.0 ** ((r_b - r_a) / scale))


# ---------- ELO FUNCTIONS ----------


def record_match_basic(
    p1: str,
    p2: str,
    winner: str,
    date: Optional[datetime] = None,
    tournament: str = "",
    surface: str = "",
    rnd: str = ""
) -> None:
    """
    Apply a single match result to the database using the basic rules:
    - Everyone starts at 500
    - Win = +5 rating and +1 win
    - Loss = -5 rating and +1 loss
    Also appends a RatingEvent to each player's history.

    Args:
        p1: Player 1's display name (as in CSV).
        p2: Player 2's display name (as in CSV).
        winner: Winner's display name; MUST equal p1 or p2.
        date: Match date; if None or invalid, defaults to datetime.min.
        tournament: Tournament name (CSV 'Tournament').
        surface: Court surface (CSV 'Surface').
        rnd: Round label (CSV 'Round').

    Raises:
        ValueError: If 'winner' is not exactly equal to p1 or p2 after stripping.
    """
    p1 = p1.strip()
    p2 = p2.strip()
    winner = winner.strip()
    a = ensure_player(p1)
    b = ensure_player(p2)
    safe_date = date or datetime.min

    if winner == p1:
        a.elo += WIN_POINTS; a.wins += 1
        b.elo += LOSS_POINTS; b.losses += 1
        a.history.append(RatingEvent(safe_date, b.name, "W", WIN_POINTS, a.elo, (tournament, surface, rnd)))
        b.history.append(RatingEvent(safe_date, a.name, "L", LOSS_POINTS, b.elo, (tournament, surface, rnd)))
    elif winner == p2:
        b.elo += WIN_POINTS; b.wins += 1
        a.elo += LOSS_POINTS; a.losses += 1
        b.history.append(RatingEvent(safe_date, a.name, "W", WIN_POINTS, b.elo, (tournament, surface, rnd)))
        a.history.append(RatingEvent(safe_date, b.name, "L", LOSS_POINTS, a.elo, (tournament, surface, rnd)))
    else:
        raise ValueError("winner must equal p1 or p2")




def record_match_elo(
    p1: str,
    p2: str,
    winner: str,
    date: Optional[datetime] = None,
    tournament: str = "",
    surface: str = "",
    rnd: str = "",
    K: float = K_DEFAULT,
    scale: float = ELO_SCALE
) -> None:
    """
    Proper Elo update:
      R_new = R_old + K * (S - E)
    S = 1 if win, 0 if loss. E = expected_score.
    """
    p1 = p1.strip(); p2 = p2.strip(); winner = winner.strip()
    a = ensure_player(p1); b = ensure_player(p2)
    safe_date = date or datetime.min

    Ea = expected_score(a.elo, b.elo, scale)
    Eb = 1.0 - Ea

    if winner == p1:
        Sa, Sb = 1.0, 0.0
    elif winner == p2:
        Sa, Sb = 0.0, 1.0
    else:
        raise ValueError("winner must equal p1 or p2")

    delta_a = K * (Sa - Ea)
    delta_b = K * (Sb - Eb)

    a.elo += delta_a; b.elo += delta_b
    if Sa == 1.0:
        a.wins += 1; b.losses += 1
        a.history.append(RatingEvent(safe_date, b.name, "W", int(round(delta_a)), int(round(a.elo)), (tournament, surface, rnd)))
        b.history.append(RatingEvent(safe_date, a.name, "L", int(round(delta_b)), int(round(b.elo)), (tournament, surface, rnd)))
    else:
        b.wins += 1; a.losses += 1
        b.history.append(RatingEvent(safe_date, a.name, "W", int(round(delta_b)), int(round(b.elo)), (tournament, surface, rnd)))
        a.history.append(RatingEvent(safe_date, b.name, "L", int(round(delta_a)), int(round(a.elo)), (tournament, surface, rnd)))



# ---------- CSV loader ----------

def load_csv_basic(path: str, limit_rows: Optional[int] = None) -> None:
    """
    Load and apply matches from the big CSV using the basic +5/-5 rules.

    CSV columns expected:
        "Tournament","Date","Series","Court","Surface","Round","Best of",
        "Player_1","Player_2","Winner","Rank_1","Rank_2","Pts_1","Pts_2",
        "Odd_1","Odd_2","Score"

    Only these are used by this loader:
        "Tournament", "Date", "Surface", "Round", "Player_1", "Player_2", "Winner"

    Args:
        path: Filesystem path to the CSV.
        limit_rows: If provided, only process the first N rows.

    Side effects:
        Mutates players_db by updating (or creating) Player entries and appending history events.
    """
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in _first_n(reader, limit_rows):
            # Extract and sanitize
            p1 = (row.get("Player_1") or "").strip()
            p2 = (row.get("Player_2") or "").strip()
            winner = (row.get("Winner") or "").strip()
            if not p1 or not p2 or not winner:
                # Skip malformed rows
                continue

            date = parse_iso_date(row.get("Date") or "")
            tournament = (row.get("Tournament") or "").strip()
            surface = (row.get("Surface") or "").strip()
            rnd = (row.get("Round") or "").strip()

            # Apply the match
            record_match_basic(
                p1, p2, winner,
                date=date,
                tournament=tournament,
                surface=surface,
                rnd=rnd
            )


def load_csv_elo(path: str, limit_rows: Optional[int] = None, K: float = K_DEFAULT) -> None:
    """
    Same as load_csv_basic but applies proper Elo updates.
    """
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in _first_n(reader, limit_rows):
            p1 = (row.get("Player_1") or "").strip()
            p2 = (row.get("Player_2") or "").strip()
            winner = (row.get("Winner") or "").strip()
            if not p1 or not p2 or not winner:
                continue
            date = parse_iso_date(row.get("Date") or "")
            tournament = (row.get("Tournament") or "").strip()
            surface = (row.get("Surface") or "").strip()
            rnd = (row.get("Round") or "").strip()

            record_match_elo(
                p1, p2, winner,
                date=date, tournament=tournament, surface=surface, rnd=rnd,
                K=K
            )





# ---------- Utilities & tests ----------

def top_n(n: int = 10) -> List[Player]:
    """
    Return the top-n players by current Elo.

    Args:
        n: Number of players to return.

    Returns:
        A list of Player objects sorted by Elo descending, length <= n.
    """
    return sorted(players_db.values(), key=lambda p: p.elo, reverse=True)[:n]

def reset_players() -> None:
    """
    Clear the in-memory player database (useful for tests).
    """
    players_db.clear()





# ---------- Saving Loading Exporting----------

def save_players_db(path: str = "players_db.json") -> None:
    """
    Save the entire players_db (players + history) to a JSON file.
    """
    serializable = {name: asdict(player) for name, player in players_db.items()}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serializable, f, ensure_ascii=False, indent=2, default=str)
    print(f"✅ Saved {len(players_db)} players to {path}")

def load_players_db(path: str = "players_db.json") -> None:
    """
    Load players_db from a JSON file.
    """
    global players_db
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    players_db = {}
    for name, pdata in raw.items():
        history = [
            RatingEvent(
                date=datetime.fromisoformat(ev["date"]) if ev["date"] != "0001-01-01 00:00:00" else datetime.min,
                opponent=ev["opponent"],
                result=ev["result"],
                delta=ev["delta"],
                new_elo=ev["new_elo"],
                meta=tuple(ev["meta"])
            )
            for ev in pdata["history"]
        ]
        players_db[name] = Player(
            name=pdata["name"],
            elo=pdata["elo"],
            wins=pdata["wins"],
            losses=pdata["losses"],
            history=history
        )
    print(f"✅ Loaded {len(players_db)} players from {path}")


    def export_players_to_csv(path: str) -> None:
        with open(path, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["Name", "Elo", "Wins", "Losses", "MatchesPlayed"])
            for p in players_db.values():
                writer.writerow([p.name, p.elo, p.wins, p.losses, p.wins + p.losses])
        print(f"✅ Exported {len(players_db)} players to {path}")


In [44]:
reset_players()
load_csv_elo("atp_tennis_clean.csv", K=32)
for p in top_n(10):
    print(p.name, round(p.elo,2), p.wins, p.losses)

Sinner J. 1213.63 240 73
Djokovic N. 1093.56 1027 193
Alcaraz C. 1021.2 187 51
Federer R. 997.86 962 160
Soderling R. 989.77 268 146
Zverev A. 983.29 423 180
Medvedev D. 942.19 342 142
Del Potro J.M. 931.33 399 153
Nadal R. 919.96 997 197
Kyrgios N. 919.01 186 93


In [46]:
export_players_to_csv("data28082025_properELO.csv")
save_players_db("data28082025_properELO.json")

✅ Exported 1390 players to data28082025_properELO.csv
✅ Saved 1390 players to data28082025_properELO.json
